## Distributed Data Systems Project
#### Student Names: Daragh Brady (D00255640), James Lawrence (D00257516), Michael Sanei (D00260668)
#### Project Name: Alcohol Consumption

## DATA EXPLORATION

In [4]:
import pandas as pd
from sqlalchemy import create_engine

load data

In [5]:
df = pd.read_csv("student_data.csv")
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel_y,freetime_y,goout_y,Dalc_y,Walc_y,health_y,absences_y,G1_y,G2_y,G3_y
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377,MS,F,18,U,LE3,T,3,1,teacher,services,...,4,3,4,1,1,1,4,15,15,16
378,MS,F,18,U,GT3,T,1,1,other,other,...,3,4,4,2,2,5,3,7,8,7
379,MS,F,18,U,GT3,T,1,1,other,other,...,1,1,1,1,1,5,6,11,12,9
380,MS,M,17,U,LE3,T,3,1,services,services,...,2,4,5,3,4,2,6,10,10,10


In [6]:
df['romantic_y'].unique()

array(['no', 'yes'], dtype=object)

check missing values

In [7]:
print(df.isnull().sum())

school          0
sex             0
age             0
address         0
famsize         0
Pstatus         0
Medu            0
Fedu            0
Mjob            0
Fjob            0
reason          0
guardian_x      0
traveltime_x    0
studytime_x     0
failures_x      0
schoolsup_x     0
famsup_x        0
paid_x          0
activities_x    0
nursery         0
higher_x        0
internet        0
romantic_x      0
famrel_x        0
freetime_x      0
goout_x         0
Dalc_x          0
Walc_x          0
health_x        0
absences_x      0
G1_x            0
G2_x            0
G3_x            0
guardian_y      0
traveltime_y    0
studytime_y     0
failures_y      0
schoolsup_y     0
famsup_y        0
paid_y          0
activities_y    0
higher_y        0
romantic_y      0
famrel_y        0
freetime_y      0
goout_y         0
Dalc_y          0
Walc_y          0
health_y        0
absences_y      0
G1_y            0
G2_y            0
G3_y            0
dtype: int64


##### from our schema we have 5 diferent tables we need to create...

In [8]:
print(df.columns)

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian_x', 'traveltime_x', 'studytime_x',
       'failures_x', 'schoolsup_x', 'famsup_x', 'paid_x', 'activities_x',
       'nursery', 'higher_x', 'internet', 'romantic_x', 'famrel_x',
       'freetime_x', 'goout_x', 'Dalc_x', 'Walc_x', 'health_x', 'absences_x',
       'G1_x', 'G2_x', 'G3_x', 'guardian_y', 'traveltime_y', 'studytime_y',
       'failures_y', 'schoolsup_y', 'famsup_y', 'paid_y', 'activities_y',
       'higher_y', 'romantic_y', 'famrel_y', 'freetime_y', 'goout_y', 'Dalc_y',
       'Walc_y', 'health_y', 'absences_y', 'G1_y', 'G2_y', 'G3_y'],
      dtype='object')


# creating fact and dimension tables


In [9]:
# Student Demographics
student_demographics = df[['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'romantic_x']].drop_duplicates()
student_demographics['DemoID'] = range(1, len(student_demographics) + 1)

# Support Info
support_info = df[['schoolsup_x', 'famsup_x', 'paid_x', 'activities_x', 'nursery', 'higher_x', 'internet']].drop_duplicates()
support_info['SupID'] = range(1, len(support_info) + 1)

# Alcohol Consumption
alcohol_consumption = df[['Dalc_x', 'Walc_x']].drop_duplicates()
alcohol_consumption['AlcID'] = range(1, len(alcohol_consumption) + 1)

# Parent Info
parent_info = df[['Medu', 'Fedu', 'Mjob', 'Fjob', 'guardian_x']].drop_duplicates()
parent_info['ParentID'] = range(1, len(parent_info) + 1)

# Family Environment
family_environment = df[['famrel_x', 'freetime_x', 'goout_x']].drop_duplicates()
family_environment['FamID'] = range(1, len(family_environment) + 1)

# Health Info
health_info = df[['health_x']].drop_duplicates()
health_info['HealthID'] = range(1, len(health_info) + 1)


In [10]:
# Merge foreign keys into the fact table
fact_table = df.merge(student_demographics, on=['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'romantic_x']) \
               .merge(support_info, on=['schoolsup_x', 'famsup_x', 'paid_x', 'activities_x', 'nursery', 'higher_x', 'internet']) \
               .merge(alcohol_consumption, on=['Dalc_x', 'Walc_x']) \
               .merge(parent_info, on=['Medu', 'Fedu', 'Mjob', 'Fjob', 'guardian_x']) \
               .merge(family_environment, on=['famrel_x', 'freetime_x', 'goout_x']) \
               .merge(health_info, on=['health_x'])

# Select columns for fact table
fact_table = fact_table[['DemoID', 'ParentID', 'SupID', 'FamID', 'AlcID', 'HealthID', 'G1_x', 'G2_x', 'G3_x', 'absences_x', 'failures_x', 'studytime_x', 'traveltime_x']]


In [12]:
# Create a connection to MySQL
engine = create_engine('mysql+pymysql://root@localhost/dds_ca')


# Test connection
engine.connect()


In [14]:
student_demographics.to_sql('Student_Demographics', engine, if_exists='replace', index=False)
support_info.to_sql('Support_Info', engine, if_exists='replace', index=False)
alcohol_consumption.to_sql('Alcohol_Consumption', engine, if_exists='replace', index=False)
parent_info.to_sql('Parent_Info', engine, if_exists='replace', index=False)
family_environment.to_sql('Family_Environment', engine, if_exists='replace', index=False)
health_info.to_sql('Health_Info', engine, if_exists='replace', index=False)
fact_table.to_sql('student_performance', engine, if_exists='replace', index=False)

C:\Users\misam\AppData\Local\Temp\ipykernel_5460\3425083010.py:1: UserWarning: The provided table name 'Student_Demographics' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  student_demographics.to_sql('Student_Demographics', engine, if_exists='replace', index=False)
C:\Users\misam\AppData\Local\Temp\ipykernel_5460\3425083010.py:2: UserWarning: The provided table name 'Support_Info' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  support_info.to_sql('Support_Info', engine, if_exists='replace', index=False)
C:\Users\misam\AppData\Local\Temp\ipykernel_5460\3425083010.py:3: UserWarning: The provided table name 'Alcohol_Consumption' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  alc

382